In [7]:
# all credit to rami younes and philippe beliveau for original version

import networkx as nx
import os, glob, json
import polars as pl
from tqdm.auto import tqdm

# src_hash = "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"
# path = "./Data/TOSHI 24-03-24 3j"

# src_hash = "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"
# path = "./Data/BRETT 24-03-20 3j/"

COL_ORDERS = ['hash', 'source', 'from', 'to', 'value', 'symbol', "sub_type"]

hashes_to_parse = [ # can glob split this
    ["./Data/TOSHI 24-03-24 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
    ["./Data/BRETT 24-03-20 3j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"]
]


for path, src_hash in tqdm(hashes_to_parse):

    main = (
        pl.read_excel(
            f"{path}/TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx",

            read_options=dict(
                columns = ['hash', 'from', 'to', 'value'],
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.lit("ETH").alias("symbol"),
            pl.lit("main").alias("source"),
            pl.lit("main").alias("sub_type")
        )
    )

    internal = (
        pl.read_excel(
            f"{path}/TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx",
            read_options=dict(
                columns = ['hash', 'from', 'to', 'value'] + ["callType"],
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.lit("ETH").alias("symbol"),
            pl.lit("internal").alias("source"),
            pl.col("callType").alias("sub_type")
        )
        .filter(
            pl.col("value") != "0",
            pl.col("sub_type") == "call"
        )
    )


    # https://ethereum.stackexchange.com/questions/65740/whats-the-reason-erc-20-standard-allows-0-value-transfer
    erc = (
        pl.read_excel(
            f"{path}/TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx",
            read_options=dict(
                columns = ['hash',  'from', 'to', 'tokenSymbol', 'value'] + ["tx_type"],
                # ignore_errors=True,
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.col("tokenSymbol").alias("symbol"),
            pl.col("tx_type").alias("sub_type"),
            pl.lit("erc").alias("source")
        )
        .filter(
            pl.col("value") != "0"
        )
    )

    erc_types = erc.select("hash", "from", "to", )

    erc_traced_replaces = {
        "Tx hash": "hash",
        "Tx type": "type",
        "value ETH ajusted"    : "ETH_adj"
    }

    erc_traced = (
        pl.read_excel(
            f"{path}/TX_ERC_Traced_Cleaned_{src_hash}.xlsx",
            read_options=dict(
                columns = ['Tx hash', 'Tx type', 'ETH', 'value ETH ajusted'],
                ignore_errors=True,
            )
        )
        .with_columns([
            pl.col(k).alias(v)
            for k, v in erc_traced_replaces.items()
        ])
        .drop(erc_traced_replaces.keys())
    )

    df = pl.concat([
        main.select(COL_ORDERS), 
        internal.select(COL_ORDERS), 
        erc.select(COL_ORDERS)
    ]).sort("hash", "source", "from", "to")

    df = (
        df
        .with_columns(
            pl.col("value").cast(float),
            *[
                pl.col(c).str.to_lowercase().alias(c)
                for c in ["from", "to"]
            ]
        )
        .with_columns(
            pl.col("symbol") # the algo tags WETH only, so assuming WETH == ETH
            .str.replace("WETH", "ETH")
            .str.replace("ETH", "WETH")
        )
    )

    df = df.join(erc_traced, on="hash")

    df.write_parquet(f"./data/{src_hash}.pqt")

  0%|          | 0/2 [00:00<?, ?it/s]

ModuleNotFoundError: required package 'fastexcel' not found.
Please install using the command `pip install fastexcel`.

In [3]:
!pip show polars

Name: polars
Version: 1.2.1
Summary: Blazingly fast DataFrame library
Home-page: None
Author: None
Author-email: Ritchie Vink <ritchie46@gmail.com>
License: None
Location: /home/helen/miniconda3/lib/python3.8/site-packages
Requires: 
Required-by: 


In [6]:
# all credit to rami younes and philippe beliveau for original version

import networkx as nx
import os, glob, json
import polars as pl
from tqdm.auto import tqdm

# src_hash = "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"
# path = "./Data/TOSHI 24-03-24 3j"

# src_hash = "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"
# path = "./Data/BRETT 24-03-20 3j/"

COL_ORDERS = ['hash', 'source', 'from', 'to', 'value', 'symbol', "sub_type"]

hashes_to_parse = [ # can glob split this
    ["./Data/TOSHI 24-03-24 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
    ["./Data/BRETT 24-03-20 3j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"]
]


for path, src_hash in tqdm(hashes_to_parse):

    main = (
        pl.read_excel(
            f"{path}/TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx",

            read_options=dict(
                columns = ['hash', 'from', 'to', 'value'],
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.lit("ETH").alias("symbol"),
            pl.lit("main").alias("source"),
            pl.lit("main").alias("sub_type")
        )
    )

    internal = (
        pl.read_excel(
            f"{path}/TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx",
            read_options=dict(
                columns = ['hash', 'from', 'to', 'value'] + ["callType"],
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.lit("ETH").alias("symbol"),
            pl.lit("internal").alias("source"),
            pl.col("callType").alias("sub_type")
        )
        .filter(
            pl.col("value") != "0",
            pl.col("sub_type") == "call"
        )
    )


    # https://ethereum.stackexchange.com/questions/65740/whats-the-reason-erc-20-standard-allows-0-value-transfer
    erc = (
        pl.read_excel(
            f"{path}/TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx",
            read_options=dict(
                columns = ['hash',  'from', 'to', 'tokenSymbol', 'value'] + ["tx_type"],
                # ignore_errors=True,
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.col("tokenSymbol").alias("symbol"),
            pl.col("tx_type").alias("sub_type"),
            pl.lit("erc").alias("source")
        )
        .filter(
            pl.col("value") != "0"
        )
    )

    erc_types = erc.select("hash", "from", "to", )

    erc_traced_replaces = {
        "Tx hash": "hash",
        "Tx type": "type",
        "value ETH ajusted"    : "ETH_adj"
    }

    erc_traced = (
        pl.read_excel(
            f"{path}/TX_ERC_Traced_Cleaned_{src_hash}.xlsx",
            read_options=dict(
                columns = ['Tx hash', 'Tx type', 'ETH', 'value ETH ajusted'],
                ignore_errors=True,
            )
        )
        .with_columns([
            pl.col(k).alias(v)
            for k, v in erc_traced_replaces.items()
        ])
        .drop(erc_traced_replaces.keys())
    )

    df = pl.concat([
        main.select(COL_ORDERS), 
        internal.select(COL_ORDERS), 
        erc.select(COL_ORDERS)
    ]).sort("hash", "source", "from", "to")

    df = (
        df
        .with_columns(
            pl.col("value").cast(float),
            *[
                pl.col(c).str.to_lowercase().alias(c)
                for c in ["from", "to"]
            ]
        )
        .with_columns(
            pl.col("symbol") # the algo tags WETH only, so assuming WETH == ETH
            .str.replace("WETH", "ETH")
            .str.replace("ETH", "WETH")
        )
    )

    df = df.join(erc_traced, on="hash")

    df.write_parquet(f"./data/{src_hash}.pqt")

  0%|          | 0/2 [00:00<?, ?it/s]

ModuleNotFoundError: required package 'fastexcel' not found.
Please install using the command `pip install fastexcel`.

In [4]:
import polars as pl

# Check available functions in Polars
dir(pl)

['Array',
 'Binary',
 'Boolean',
 'Categorical',
 'CompatLevel',
 'Config',
 'DataFrame',
 'DataType',
 'Date',
 'Datetime',
 'Decimal',
 'Duration',
 'Enum',
 'Expr',
 'Field',
 'Float32',
 'Float64',
 'Int16',
 'Int32',
 'Int64',
 'Int8',
 'LazyFrame',
 'List',
 'Null',
 'Object',
 'SQLContext',
 'Schema',
 'Series',
 'String',
 'StringCache',
 'Struct',
 'Time',
 'UInt16',
 'UInt32',
 'UInt64',
 'UInt8',
 'Unknown',
 'Utf8',
 '__all__',
 '__annotations__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__getattr__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__register_startup_deps',
 '__spec__',
 '__version__',
 '_cpu_check',
 '_reexport',
 '_typing',
 '_utils',
 'align_frames',
 'all',
 'all_horizontal',
 'any',
 'any_horizontal',
 'api',
 'approx_n_unique',
 'arange',
 'arctan2',
 'arctan2d',
 'arg_sort_by',
 'arg_where',
 'build_info',
 'business_day_count',
 'coalesce',
 'col',
 'collect_all',
 'collect_all_async',
 'concat',
 'concat_list',
 'concat

In [1]:
# all credit to rami younes and philippe beliveau for original version

import networkx as nx
import os, glob, json
import polars as pl
from tqdm.auto import tqdm

# src_hash = "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"
# path = "./Data/TOSHI 24-03-24 3j"

# src_hash = "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"
# path = "./Data/BRETT 24-03-20 3j/"

COL_ORDERS = ['hash', 'source', 'from', 'to', 'value', 'symbol', "sub_type"]

hashes_to_parse = [ # can glob split this
    ["./Data/TOSHI 24-03-24 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
    ["./Data/BRETT 24-03-20 3j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"]
]


for path, src_hash in tqdm(hashes_to_parse):

    main = (
        pl.read_excel(
            f"{path}/TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx",

            read_options=dict(
                columns = ['hash', 'from', 'to', 'value'],
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.lit("ETH").alias("symbol"),
            pl.lit("main").alias("source"),
            pl.lit("main").alias("sub_type")
        )
    )

    internal = (
        pl.read_excel(
            f"{path}/TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx",
            read_options=dict(
                columns = ['hash', 'from', 'to', 'value'] + ["callType"],
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.lit("ETH").alias("symbol"),
            pl.lit("internal").alias("source"),
            pl.col("callType").alias("sub_type")
        )
        .filter(
            pl.col("value") != "0",
            pl.col("sub_type") == "call"
        )
    )


    # https://ethereum.stackexchange.com/questions/65740/whats-the-reason-erc-20-standard-allows-0-value-transfer
    erc = (
        pl.read_excel(
            f"{path}/TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx",
            read_options=dict(
                columns = ['hash',  'from', 'to', 'tokenSymbol', 'value'] + ["tx_type"],
                # ignore_errors=True,
                dtypes={"value":str},
            )
        )
        .with_columns(
            pl.col("tokenSymbol").alias("symbol"),
            pl.col("tx_type").alias("sub_type"),
            pl.lit("erc").alias("source")
        )
        .filter(
            pl.col("value") != "0"
        )
    )

    erc_types = erc.select("hash", "from", "to", )

    erc_traced_replaces = {
        "Tx hash": "hash",
        "Tx type": "type",
        "value ETH ajusted"    : "ETH_adj"
    }

    erc_traced = (
        pl.read_excel(
            f"{path}/TX_ERC_Traced_Cleaned_{src_hash}.xlsx",
            read_options=dict(
                columns = ['Tx hash', 'Tx type', 'ETH', 'value ETH ajusted'],
                ignore_errors=True,
            )
        )
        .with_columns([
            pl.col(k).alias(v)
            for k, v in erc_traced_replaces.items()
        ])
        .drop(erc_traced_replaces.keys())
    )

    df = pl.concat([
        main.select(COL_ORDERS), 
        internal.select(COL_ORDERS), 
        erc.select(COL_ORDERS)
    ]).sort("hash", "source", "from", "to")

    df = (
        df
        .with_columns(
            pl.col("value").cast(float),
            *[
                pl.col(c).str.to_lowercase().alias(c)
                for c in ["from", "to"]
            ]
        )
        .with_columns(
            pl.col("symbol") # the algo tags WETH only, so assuming WETH == ETH
            .str.replace("WETH", "ETH")
            .str.replace("ETH", "WETH")
        )
    )

    df = df.join(erc_traced, on="hash")

    df.write_parquet(f"./data/{src_hash}.pqt")

  0%|          | 0/2 [00:00<?, ?it/s]

ModuleNotFoundError: required package 'fastexcel' not found.
Please install using the command `pip install fastexcel`.

In [6]:
df.head()

hash,source,from,to,value,symbol,sub_type,ETH,type,ETH_adj
str,str,str,str,f64,str,str,f64,str,f64
"""0x0002738472d6fe6737be63ea6067…","""erc""","""0x3fc91a3afd70395cd496c647d5a6…","""0x0000000000000000000000000000…",0.232052,"""WETH""","""ERC20""",-0.232052,"""swap""",-0.000002
"""0x0002738472d6fe6737be63ea6067…","""erc""","""0xba3f945812a83471d709bce9c3ca…","""0x3fc91a3afd70395cd496c647d5a6…",0.232052,"""WETH""","""ERC20""",-0.232052,"""swap""",-0.000002
"""0x0002738472d6fe6737be63ea6067…","""erc""","""0xdc64af1f593505b70f3b45383d96…","""0xba3f945812a83471d709bce9c3ca…",28186.0,"""BRETT""","""ERC20""",-0.232052,"""swap""",-0.000002
"""0x0002738472d6fe6737be63ea6067…","""internal""","""0x3fc91a3afd70395cd496c647d5a6…","""0xdc64af1f593505b70f3b45383d96…",0.232052,"""WETH""","""call""",-0.232052,"""swap""",-0.000002
"""0x0002738472d6fe6737be63ea6067…","""internal""","""0x4200000000000000000000000000…","""0x3fc91a3afd70395cd496c647d5a6…",0.232052,"""WETH""","""call""",-0.232052,"""swap""",-0.000002
